In [2]:
import pandas as pd
import numpy as np
import joblib
import json
import time
from pathlib import Path

model_files = [
    "xpass_event_360_model.pkl",
    "final_player_forecasting_models.pkl"
]

for file in model_files:
    path = Path(file)

    print(
        file,
        "| exists:", path.exists(),
        "| size:", path.stat().st_size if path.exists() else None
    )

print("\nTrying joblib loading...")

xpass_model = joblib.load(
    "xpass_event_360_model.pkl"
)

forecasting_models = joblib.load(
    "final_player_forecasting_models.pkl"
)

historical_master = pd.read_pickle(
    "historical_player_match_master_corrected.pkl"
)

print("\nxPass model loaded successfully")
print("xPass type:", type(xpass_model))

print("\nForecasting models loaded successfully")
print("Forecasting object type:", type(forecasting_models))

print("\nHistorical master:", historical_master.shape)

if isinstance(forecasting_models, dict):
    print("\nForecasting targets:")
    print(list(forecasting_models.keys()))

xpass_event_360_model.pkl | exists: True | size: 3499654
final_player_forecasting_models.pkl | exists: True | size: 4264900

Trying joblib loading...

xPass model loaded successfully
xPass type: <class 'sklearn.pipeline.Pipeline'>

Forecasting models loaded successfully
Forecasting object type: <class 'dict'>

Historical master: (113433, 83)

Forecasting targets:
['second_half_actions_per90', 'second_half_miscontrols_per90', 'second_half_pass_completion_rate', 'second_half_progressive_passes_per90', 'second_half_statsbomb_xg_per90']


In [3]:
match_id = 3764440

events_path = Path(
    f"open-data-master/data/events/{match_id}.json"
)

three_sixty_path = Path(
    f"open-data-master/data/three-sixty/{match_id}.json"
)

print("Events file exists:", events_path.exists())
print("360 file exists:", three_sixty_path.exists())

with open(events_path, "r", encoding="utf-8") as f:
    match_events = json.load(f)

with open(three_sixty_path, "r", encoding="utf-8") as f:
    match_360 = json.load(f)

print("\nEvent records:", len(match_events))
print("360 records:", len(match_360))

print("\nFirst event keys:")
print(match_events[0].keys())

print("\nFirst 360 record keys:")
print(match_360[0].keys())

Events file exists: True
360 file exists: True

Event records: 4160
360 records: 3914

First event keys:
dict_keys(['id', 'index', 'period', 'timestamp', 'minute', 'second', 'type', 'possession', 'possession_team', 'play_pattern', 'team', 'duration', 'tactics'])

First 360 record keys:
dict_keys(['event_uuid', 'visible_area', 'freeze_frame'])


In [4]:
three_sixty_lookup = {
    record["event_uuid"]: {
        "freeze_frame": record.get("freeze_frame"),
        "visible_area": record.get("visible_area")
    }
    for record in match_360
}

print("360 lookup entries:", len(three_sixty_lookup))

sample_event_id = match_360[0]["event_uuid"]

print("\nSample event UUID:")
print(sample_event_id)

print("\nSample 360 data:")
print(three_sixty_lookup[sample_event_id])

360 lookup entries: 3914

Sample event UUID:
23743e50-fbe4-4929-938e-9b3ab39ff4ff

Sample 360 data:
{'freeze_frame': [{'teammate': False, 'actor': False, 'keeper': False, 'location': [52.273950315181956, 38.42847753920154]}, {'teammate': False, 'actor': False, 'keeper': False, 'location': [54.32713847631563, 47.166411361696724]}, {'teammate': False, 'actor': False, 'keeper': False, 'location': [55.867029597165875, 31.488912939271692]}, {'teammate': False, 'actor': False, 'keeper': False, 'location': [58.946811838866346, 55.132085341341906]}, {'teammate': True, 'actor': True, 'keeper': False, 'location': [61.0, 40.099998474121094]}, {'teammate': True, 'actor': False, 'keeper': False, 'location': [61.25664852014171, 56.417276328163084]}, {'teammate': True, 'actor': False, 'keeper': False, 'location': [61.76994556042512, 30.461066072962247]}], 'visible_area': [0.0, 50.96006313698502, 0.0, 45.50103554950569, 3.342087731980023, 26.083370794549182, 19.892844407502402, 29.845733228438647, 21.

In [5]:
replay_events = []

for event in match_events:
    event_id = event["id"]

    spatial = three_sixty_lookup.get(
        event_id,
        {}
    )

    replay_events.append({
        "event_id": event_id,
        "index": event.get("index"),
        "period": event.get("period"),
        "timestamp": event.get("timestamp"),
        "minute": event.get("minute"),
        "second": event.get("second"),
        "type": event.get("type", {}).get("name"),
        "team": event.get("team", {}).get("name"),
        "player_id": (
            event.get("player", {}).get("id")
            if event.get("player")
            else None
        ),
        "player_name": (
            event.get("player", {}).get("name")
            if event.get("player")
            else None
        ),
        "raw_event": event,
        "freeze_frame": spatial.get("freeze_frame"),
        "visible_area": spatial.get("visible_area")
    })

replay_events = sorted(
    replay_events,
    key=lambda x: x["index"]
)

print("Replay events:", len(replay_events))

events_with_360 = sum(
    event["freeze_frame"] is not None
    for event in replay_events
)

print("Events with 360 context:", events_with_360)

print("\nFirst 5 replay events:")

for event in replay_events[:5]:
    print(
        event["index"],
        event["minute"],
        event["second"],
        event["type"],
        event["team"],
        event["player_name"],
        "360:",
        event["freeze_frame"] is not None
    )

Replay events: 4160
Events with 360 context: 3914

First 5 replay events:
1 0 0 Starting XI Barcelona None 360: False
2 0 0 Starting XI Elche None 360: False
3 0 0 Half Start Barcelona None 360: False
4 0 0 Half Start Elche None 360: False
5 0 0 Pass Elche Pere Milla Peña 360: True


In [6]:
from collections import defaultdict

def create_player_state():
    return {
        "team": None,
        "player_name": None,
        "actions": 0,
        "pass_attempts": 0,
        "completed_passes": 0,
        "progressive_passes": 0,
        "expected_completions": 0.0,
        "shots": 0,
        "goals": 0,
        "statsbomb_xg": 0.0,
        "carries": 0,
        "miscontrols": 0,
        "interceptions": 0,
        "ball_recoveries": 0,
        "dribbles": 0,
        "duels": 0,
        "pressures": 0,
        "shot_assists": 0
    }

live_player_state = defaultdict(
    create_player_state
)

print(
    "Initial players in live state:",
    len(live_player_state)
)

Initial players in live state: 0


In [7]:
def update_player_state(event, live_player_state):
    player_id = event.get("player_id")

    if player_id is None:
        return

    state = live_player_state[player_id]

    state["player_name"] = event.get("player_name")
    state["team"] = event.get("team")
    state["actions"] += 1

    raw_event = event["raw_event"]
    event_type = event.get("type")

    if event_type == "Pass":
        state["pass_attempts"] += 1

        pass_data = raw_event.get("pass", {})

        if pass_data.get("outcome") is None:
            state["completed_passes"] += 1

        end_location = pass_data.get("end_location")
        start_location = raw_event.get("location")

        if (
            start_location is not None
            and end_location is not None
        ):
            start_x, start_y = start_location[:2]
            end_x, end_y = end_location[:2]

            start_goal_distance = np.sqrt(
                (120 - start_x) ** 2
                + (40 - start_y) ** 2
            )

            end_goal_distance = np.sqrt(
                (120 - end_x) ** 2
                + (40 - end_y) ** 2
            )

            if start_goal_distance > 0:
                reduction_pct = (
                    (
                        start_goal_distance
                        - end_goal_distance
                    )
                    / start_goal_distance
                ) * 100

                if reduction_pct >= 25:
                    state["progressive_passes"] += 1

        if pass_data.get("shot_assist") is True:
            state["shot_assists"] += 1

    elif event_type == "Shot":
        state["shots"] += 1

        shot_data = raw_event.get("shot", {})

        state["statsbomb_xg"] += float(
            shot_data.get(
                "statsbomb_xg",
                0
            )
            or 0
        )

        if (
            shot_data.get("outcome", {})
            .get("name")
            == "Goal"
        ):
            state["goals"] += 1

    elif event_type == "Carry":
        state["carries"] += 1

    elif event_type == "Miscontrol":
        state["miscontrols"] += 1

    elif event_type == "Interception":
        state["interceptions"] += 1

    elif event_type == "Ball Recovery":
        state["ball_recoveries"] += 1

    elif event_type == "Dribble":
        state["dribbles"] += 1

    elif event_type == "Duel":
        state["duels"] += 1

    elif event_type == "Pressure":
        state["pressures"] += 1

In [8]:
live_player_state = defaultdict(
    create_player_state
)

for event in replay_events[:100]:
    update_player_state(
        event,
        live_player_state
    )

print(
    "Players observed after 100 events:",
    len(live_player_state)
)

sample_states = list(
    live_player_state.items()
)[:5]

for player_id, state in sample_states:
    print(
        "\nPlayer ID:",
        player_id
    )
    print(state)

Players observed after 100 events: 18

Player ID: 12072
{'team': 'Elche', 'player_name': 'Pere Milla Peña', 'actions': 4, 'pass_attempts': 1, 'completed_passes': 1, 'progressive_passes': 0, 'expected_completions': 0.0, 'shots': 0, 'goals': 0, 'statsbomb_xg': 0.0, 'carries': 0, 'miscontrols': 0, 'interceptions': 0, 'ball_recoveries': 0, 'dribbles': 0, 'duels': 0, 'pressures': 1, 'shot_assists': 0}

Player ID: 24517
{'team': 'Elche', 'player_name': 'José Raúl Gutiérrez Parejo', 'actions': 4, 'pass_attempts': 1, 'completed_passes': 1, 'progressive_passes': 0, 'expected_completions': 0.0, 'shots': 0, 'goals': 0, 'statsbomb_xg': 0.0, 'carries': 1, 'miscontrols': 0, 'interceptions': 0, 'ball_recoveries': 0, 'dribbles': 0, 'duels': 0, 'pressures': 1, 'shot_assists': 0}

Player ID: 24169
{'team': 'Elche', 'player_name': 'Gonzalo Cacicedo Verdú', 'actions': 9, 'pass_attempts': 3, 'completed_passes': 3, 'progressive_passes': 1, 'expected_completions': 0.0, 'shots': 0, 'goals': 0, 'statsbomb_xg':

In [9]:
def create_live_xpass_features(event):
    raw_event = event["raw_event"]
    pass_data = raw_event.get("pass", {})
    freeze_frame = event.get("freeze_frame")

    start_location = raw_event.get("location")

    if (
        start_location is None
        or freeze_frame is None
    ):
        return None

    start_x, start_y = start_location[:2]

    actor = next(
        (
            player for player in freeze_frame
            if player.get("actor") is True
        ),
        None
    )

    if actor is None:
        return None

    actor_location = actor.get("location")

    if actor_location is None:
        return None

    opponents = [
        player for player in freeze_frame
        if player.get("teammate") is False
        and player.get("location") is not None
    ]

    teammates = [
        player for player in freeze_frame
        if player.get("teammate") is True
        and player.get("actor") is False
        and player.get("location") is not None
    ]

    def distance(location_1, location_2):
        return np.sqrt(
            (location_1[0] - location_2[0]) ** 2
            + (location_1[1] - location_2[1]) ** 2
        )

    opponent_distances = [
        distance(
            actor_location,
            player["location"]
        )
        for player in opponents
    ]

    teammate_distances = [
        distance(
            actor_location,
            player["location"]
        )
        for player in teammates
    ]

    nearest_opponent_distance = (
        min(opponent_distances)
        if opponent_distances
        else np.nan
    )

    nearest_teammate_distance = (
        min(teammate_distances)
        if teammate_distances
        else np.nan
    )

    nearby_opponents_5 = sum(
        d <= 5
        for d in opponent_distances
    )

    nearby_teammates_5 = sum(
        d <= 5
        for d in teammate_distances
    )

    nearby_opponents_10 = sum(
        d <= 10
        for d in opponent_distances
    )

    nearby_teammates_10 = sum(
        d <= 10
        for d in teammate_distances
    )

    under_spatial_pressure = int(
        nearest_opponent_distance <= 5
        if not np.isnan(nearest_opponent_distance)
        else False
    )

    features = pd.DataFrame([{
        "pass_length": pass_data.get("length"),
        "pass_angle_degrees": np.degrees(
            pass_data.get("angle")
        ) if pass_data.get("angle") is not None else np.nan,

        "start_x": start_x,
        "start_y": start_y,

        "is_cross": int(
            pass_data.get("cross", False)
        ),

        "is_through_ball": int(
            pass_data.get("through_ball", False)
        ),

        "is_switch": int(
            pass_data.get("switch", False)
        ),

        "nearest_opponent_distance":
            nearest_opponent_distance,

        "nearest_teammate_distance":
            nearest_teammate_distance,

        "under_spatial_pressure":
            under_spatial_pressure,

        "nearby_opponents_5":
            nearby_opponents_5,

        "nearby_teammates_5":
            nearby_teammates_5,

        "local_numerical_balance_5":
            nearby_teammates_5
            - nearby_opponents_5,

        "nearby_opponents_10":
            nearby_opponents_10,

        "nearby_teammates_10":
            nearby_teammates_10,

        "local_numerical_balance_10":
            nearby_teammates_10
            - nearby_opponents_10
    }])

    return features.astype(float)

In [10]:
first_pass = next(
    event for event in replay_events
    if event["type"] == "Pass"
    and event["freeze_frame"] is not None
)

first_pass_features = create_live_xpass_features(
    first_pass
)

display(first_pass_features)

first_pass_xpass = xpass_model.predict_proba(
    first_pass_features
)[0, 1]

print(
    "\nPlayer:",
    first_pass["player_name"]
)

print(
    "xPass probability:",
    round(first_pass_xpass, 4)
)

print(
    "Actual completed:",
    first_pass["raw_event"]
    .get("pass", {})
    .get("outcome") is None
)

,pass_length,pass_angle_degrees,start_x,start_y,is_cross,is_through_ball,is_switch,nearest_opponent_distance,nearest_teammate_distance,under_spatial_pressure,nearby_opponents_5,nearby_teammates_5,local_numerical_balance_5,nearby_opponents_10,nearby_teammates_10,local_numerical_balance_10
0,15.597436,139.159646,61.0,40.1,0.0,0.0,0.0,8.884702,9.669635,0.0,0.0,0.0,0.0,2.0,1.0,-1.0



Player: Pere Milla Peña
xPass probability: 0.9975
Actual completed: True


In [11]:
def process_live_event(event, live_player_state):
    update_player_state(
        event,
        live_player_state
    )

    if event.get("type") != "Pass":
        return

    player_id = event.get("player_id")

    if player_id is None:
        return

    features = create_live_xpass_features(
        event
    )

    if features is None:
        return

    probability = xpass_model.predict_proba(
        features
    )[0, 1]

    live_player_state[
        player_id
    ]["expected_completions"] += probability

In [12]:
live_player_state = defaultdict(
    create_player_state
)

for event in replay_events[:500]:
    process_live_event(
        event,
        live_player_state
    )

live_state_df = pd.DataFrame(
    live_player_state.values()
)

live_state_df["actual_minus_expected"] = (
    live_state_df["completed_passes"]
    - live_state_df["expected_completions"]
)

live_state_df["pass_completion_rate"] = np.where(
    live_state_df["pass_attempts"] > 0,
    live_state_df["completed_passes"]
    / live_state_df["pass_attempts"],
    np.nan
)

display(
    live_state_df[
        [
            "player_name",
            "team",
            "pass_attempts",
            "completed_passes",
            "pass_completion_rate",
            "expected_completions",
            "actual_minus_expected"
        ]
    ]
    .sort_values(
        "pass_attempts",
        ascending=False
    )
    .head(10)
    .round(3)
)

,player_name,team,pass_attempts,completed_passes,pass_completion_rate,expected_completions,actual_minus_expected
7,Gerard Piqué Bernabéu,Barcelona,27,24,0.889,23.242001,0.758
6,Samuel Yves Umtiti,Barcelona,17,17,1.000,16.733000,0.267
17,Jordi Alba Ramos,Barcelona,14,13,0.929,11.793000,1.207
18,Óscar Mingueza García,Barcelona,14,13,0.929,12.156000,0.844
10,Francisco António Machado Mota de Castro Trincão,Barcelona,12,12,1.000,11.447000,0.553
16,Pedro González López,Barcelona,10,9,0.900,9.218000,-0.218
9,Miralem Pjanić,Barcelona,8,8,1.000,6.732000,1.268
12,Lionel Andrés Messi Cuccittini,Barcelona,7,6,0.857,5.388000,0.612
8,Frenkie de Jong,Barcelona,6,6,1.000,5.494000,0.506
3,José Manuel Sánchez Guillén,Elche,6,4,0.667,4.786000,-0.786


In [13]:
def create_player_state():
    return {
        "team": None,
        "player_name": None,
        "actions": 0,
        "pass_attempts": 0,
        "completed_passes": 0,
        "progressive_passes": 0,

        "xpass_evaluated_passes": 0,
        "xpass_actual_completions": 0,
        "expected_completions": 0.0,

        "shots": 0,
        "goals": 0,
        "statsbomb_xg": 0.0,
        "carries": 0,
        "miscontrols": 0,
        "interceptions": 0,
        "ball_recoveries": 0,
        "dribbles": 0,
        "duels": 0,
        "pressures": 0,
        "shot_assists": 0
    }

In [14]:
def process_live_event(event, live_player_state):
    update_player_state(
        event,
        live_player_state
    )

    if event.get("type") != "Pass":
        return

    player_id = event.get("player_id")

    if player_id is None:
        return

    features = create_live_xpass_features(
        event
    )

    if features is None:
        return

    probability = xpass_model.predict_proba(
        features
    )[0, 1]

    state = live_player_state[player_id]

    state["xpass_evaluated_passes"] += 1

    state["expected_completions"] += probability

    pass_data = event[
        "raw_event"
    ].get("pass", {})

    if pass_data.get("outcome") is None:
        state["xpass_actual_completions"] += 1

In [15]:
live_player_state = defaultdict(
    create_player_state
)

for event in replay_events[:500]:
    process_live_event(
        event,
        live_player_state
    )

live_state_df = pd.DataFrame(
    live_player_state.values()
)

live_state_df["xpass_actual_minus_expected"] = (
    live_state_df["xpass_actual_completions"]
    - live_state_df["expected_completions"]
)

live_state_df["xpass_coverage"] = np.where(
    live_state_df["pass_attempts"] > 0,
    live_state_df["xpass_evaluated_passes"]
    / live_state_df["pass_attempts"],
    np.nan
)

display(
    live_state_df[
        [
            "player_name",
            "team",
            "pass_attempts",
            "xpass_evaluated_passes",
            "xpass_actual_completions",
            "expected_completions",
            "xpass_actual_minus_expected",
            "xpass_coverage"
        ]
    ]
    .sort_values(
        "pass_attempts",
        ascending=False
    )
    .head(10)
    .round(3)
)

,player_name,team,pass_attempts,xpass_evaluated_passes,xpass_actual_completions,expected_completions,xpass_actual_minus_expected,xpass_coverage
7,Gerard Piqué Bernabéu,Barcelona,27,26,23,23.242001,-0.242,0.963
6,Samuel Yves Umtiti,Barcelona,17,17,17,16.733000,0.267,1.000
17,Jordi Alba Ramos,Barcelona,14,13,12,11.793000,0.207,0.929
18,Óscar Mingueza García,Barcelona,14,14,13,12.156000,0.844,1.000
10,Francisco António Machado Mota de Castro Trincão,Barcelona,12,12,12,11.447000,0.553,1.000
16,Pedro González López,Barcelona,10,10,9,9.218000,-0.218,1.000
9,Miralem Pjanić,Barcelona,8,7,7,6.732000,0.268,0.875
12,Lionel Andrés Messi Cuccittini,Barcelona,7,7,6,5.388000,0.612,1.000
8,Frenkie de Jong,Barcelona,6,6,6,5.494000,0.506,1.000
3,José Manuel Sánchez Guillén,Elche,6,6,4,4.786000,-0.786,1.000


In [16]:
match_history = historical_master[
    historical_master["match_id"] == match_id
].copy()

print(
    "Historical rows for match:",
    len(match_history)
)

print(
    "Players:",
    match_history["player_id"].nunique()
)

history_columns = [
    "player_id",
    "player_name",
    "previous_matches_available",
    "previous_reliable_pass_attempts_per90",
    "previous_reliable_progressive_passes_per90",
    "previous_reliable_carries_per90",
    "previous_reliable_shots_per90",
    "previous_reliable_miscontrols_per90",
    "previous_reliable_shot_assists_per90"
]

display(
    match_history[
        history_columns
    ]
    .head(15)
    .round(3)
)

Historical rows for match: 0
Players: 0


,player_id,player_name,previous_matches_available,previous_reliable_pass_attempts_per90,previous_reliable_progressive_passes_per90,previous_reliable_carries_per90,previous_reliable_shots_per90,previous_reliable_miscontrols_per90,previous_reliable_shot_assists_per90


In [17]:
historical_match_ids = set(
    historical_master["match_id"]
    .dropna()
    .astype(int)
    .unique()
)

three_sixty_folder = Path(
    "open-data-master/data/three-sixty"
)

three_sixty_match_ids = set(
    int(file.stem)
    for file in three_sixty_folder.glob("*.json")
)

common_match_ids = sorted(
    historical_match_ids
    & three_sixty_match_ids
)

print(
    "Historical matches:",
    len(historical_match_ids)
)

print(
    "360 matches:",
    len(three_sixty_match_ids)
)

print(
    "Matches available in both:",
    len(common_match_ids)
)

print(
    "\nFirst 20 common match IDs:"
)

print(
    common_match_ids[:20]
)

Historical matches: 3961
360 matches: 426
Matches available in both: 426

First 20 common match IDs:
[3764440, 3764661, 3773369, 3773372, 3773377, 3773386, 3773387, 3773403, 3773415, 3773428, 3773457, 3773466, 3773474, 3773477, 3773497, 3773523, 3773526, 3773547, 3773552, 3773565]


In [18]:
print(
    "historical match_id dtype:",
    historical_master["match_id"].dtype
)

print(
    "Current match_id type:",
    type(match_id)
)

print(
    "\nExample historical match IDs:"
)

print(
    historical_master["match_id"]
    .dropna()
    .head()
    .tolist()
)

historical match_id dtype: str
Current match_id type: <class 'int'>

Example historical match IDs:
['3837650', '3837659', '3837662', '3837680', '3837683']


In [19]:
historical_master["match_id"] = pd.to_numeric(
    historical_master["match_id"],
    errors="coerce"
).astype("Int64")

match_id = int(match_id)

match_history = historical_master[
    historical_master["match_id"] == match_id
].copy()

print(
    "Historical rows for match:",
    len(match_history)
)

print(
    "Players:",
    match_history["player_id"].nunique()
)

Historical rows for match: 32
Players: 32


In [20]:
history_columns = [
    "player_id",
    "player_name",
    "previous_matches_available",
    "previous_reliable_pass_attempts_per90",
    "previous_reliable_progressive_passes_per90",
    "previous_reliable_carries_per90",
    "previous_reliable_shots_per90",
    "previous_reliable_miscontrols_per90",
    "previous_reliable_shot_assists_per90"
]

display(
    match_history[
        history_columns
    ]
    .head(20)
    .round(3)
)

,player_id,player_name,previous_matches_available,previous_reliable_pass_attempts_per90,previous_reliable_progressive_passes_per90,previous_reliable_carries_per90,previous_reliable_shots_per90,previous_reliable_miscontrols_per90,previous_reliable_shot_assists_per90
4481,3246,Guido Marcelo Carrillo,33,41.353,3.446,41.353,3.446,3.446,0.000
15821,4447,Martin Braithwaite Christensen,70,17.457,0.000,23.276,2.909,0.000,2.909
22389,5203,Sergio Busquets i Burgos,379,146.312,25.740,130.055,0.000,2.709,5.419
22904,5211,Jordi Alba Ramos,234,84.894,10.150,69.207,0.923,0.923,0.923
23264,5213,Gerard Piqué Bernabéu,331,86.740,6.459,66.439,0.000,0.000,0.000
25179,5477,Ousmane Dembélé,89,95.566,9.886,102.156,6.591,2.197,2.197
25467,5487,Antoine Griezmann,112,35.135,1.351,39.189,1.351,0.000,0.000
25656,5492,Samuel Yves Umtiti,116,78.346,1.959,56.801,0.000,0.000,0.000
26257,5503,Lionel Andrés Messi Cuccittini,512,89.508,22.146,87.662,7.382,1.846,1.846
29135,5691,Johan Andrés Mojica Palacio,7,43.092,11.446,34.339,0.673,1.347,1.347


In [21]:
history_lookup = (
    match_history
    .set_index("player_id")
    .to_dict("index")
)

for player_id, state in live_player_state.items():

    history = history_lookup.get(
        player_id
    )

    if history is None:
        state["previous_matches_available"] = 0
        state["historical_pass_attempts_per90"] = np.nan
        state["historical_progressive_passes_per90"] = np.nan
        state["historical_carries_per90"] = np.nan
        state["historical_shots_per90"] = np.nan
        state["historical_miscontrols_per90"] = np.nan
        state["historical_shot_assists_per90"] = np.nan

        continue

    state["previous_matches_available"] = (
        history["previous_matches_available"]
    )

    state["historical_pass_attempts_per90"] = (
        history[
            "previous_reliable_pass_attempts_per90"
        ]
    )

    state["historical_progressive_passes_per90"] = (
        history[
            "previous_reliable_progressive_passes_per90"
        ]
    )

    state["historical_carries_per90"] = (
        history[
            "previous_reliable_carries_per90"
        ]
    )

    state["historical_shots_per90"] = (
        history[
            "previous_reliable_shots_per90"
        ]
    )

    state["historical_miscontrols_per90"] = (
        history[
            "previous_reliable_miscontrols_per90"
        ]
    )

    state["historical_shot_assists_per90"] = (
        history[
            "previous_reliable_shot_assists_per90"
        ]
    )

In [22]:
live_history_df = pd.DataFrame(
    live_player_state.values()
)

display(
    live_history_df[
        [
            "player_name",
            "team",
            "previous_matches_available",
            "historical_pass_attempts_per90",
            "historical_progressive_passes_per90",
            "historical_carries_per90",
            "historical_shots_per90"
        ]
    ]
    .sort_values(
        "previous_matches_available",
        ascending=False
    )
    .head(15)
    .round(3)
)

,player_name,team,previous_matches_available,historical_pass_attempts_per90,historical_progressive_passes_per90,historical_carries_per90,historical_shots_per90
12,Lionel Andrés Messi Cuccittini,Barcelona,512,89.508,22.146,87.662,7.382
7,Gerard Piqué Bernabéu,Barcelona,331,86.740,6.459,66.439,0.000
17,Jordi Alba Ramos,Barcelona,234,84.894,10.150,69.207,0.923
5,Marc-André ter Stegen,Barcelona,153,23.992,0.923,16.610,0.000
6,Samuel Yves Umtiti,Barcelona,116,78.346,1.959,56.801,0.000
15,Martin Braithwaite Christensen,Barcelona,70,17.457,0.000,23.276,2.909
9,Miralem Pjanić,Barcelona,49,118.650,0.000,101.286,2.894
8,Frenkie de Jong,Barcelona,45,74.744,5.537,57.211,0.923
14,Antonio Barragán Fernández,Elche,29,51.208,4.831,31.884,0.000
16,Pedro González López,Barcelona,21,80.952,5.952,80.952,0.000


In [23]:
last_event = replay_events[499]

elapsed_minutes = (
    last_event["minute"]
    + last_event["second"] / 60
)

print(
    "Current replay time:",
    f"{last_event['minute']}:{last_event['second']:02d}"
)

print(
    "Elapsed minutes:",
    round(elapsed_minutes, 2)
)

live_state_df = pd.DataFrame(
    live_player_state.values()
)

rate_metrics = [
    "actions",
    "pass_attempts",
    "progressive_passes",
    "carries",
    "shots",
    "miscontrols",
    "shot_assists"
]

for metric in rate_metrics:
    live_state_df[
        f"current_{metric}_per90"
    ] = (
        live_state_df[metric]
        / elapsed_minutes
        * 90
    )

live_state_df[
    "current_pass_completion_rate"
] = np.where(
    live_state_df["pass_attempts"] > 0,
    live_state_df["completed_passes"]
    / live_state_df["pass_attempts"],
    np.nan
)

display(
    live_state_df[
        [
            "player_name",
            "team",
            "current_pass_attempts_per90",
            "historical_pass_attempts_per90",
            "current_progressive_passes_per90",
            "historical_progressive_passes_per90",
            "current_carries_per90",
            "historical_carries_per90"
        ]
    ]
    .sort_values(
        "current_pass_attempts_per90",
        ascending=False
    )
    .head(15)
    .round(2)
)

Current replay time: 11:19
Elapsed minutes: 11.32


,player_name,team,current_pass_attempts_per90,historical_pass_attempts_per90,current_progressive_passes_per90,historical_progressive_passes_per90,current_carries_per90,historical_carries_per90
7,Gerard Piqué Bernabéu,Barcelona,214.73,86.74,23.86,6.46,190.87,66.44
6,Samuel Yves Umtiti,Barcelona,135.20,78.35,0.00,1.96,119.29,56.80
17,Jordi Alba Ramos,Barcelona,111.34,84.89,7.95,10.15,63.62,69.21
18,Óscar Mingueza García,Barcelona,111.34,94.39,15.91,2.45,95.43,77.23
10,Francisco António Machado Mota de Castro Trincão,Barcelona,95.43,28.72,7.95,0.00,103.39,32.83
16,Pedro González López,Barcelona,79.53,80.95,0.00,5.95,71.58,80.95
9,Miralem Pjanić,Barcelona,63.62,118.65,0.00,0.00,55.67,101.29
12,Lionel Andrés Messi Cuccittini,Barcelona,55.67,89.51,23.86,22.15,47.72,87.66
8,Frenkie de Jong,Barcelona,47.72,74.74,0.00,5.54,47.72,57.21
3,José Manuel Sánchez Guillén,Elche,47.72,NaN,7.95,NaN,31.81,NaN


In [24]:
starting_players = {}

for event in match_events:
    if event.get("type", {}).get("name") != "Starting XI":
        continue

    team_name = event["team"]["name"]

    lineup = (
        event.get("tactics", {})
        .get("lineup", [])
    )

    for player in lineup:
        player_id = player["player"]["id"]
        player_name = player["player"]["name"]

        starting_players[player_id] = {
            "player_name": player_name,
            "team": team_name,
            "start_minute": 0.0,
            "end_minute": None
        }

print(
    "Starting players:",
    len(starting_players)
)

for player_id, info in list(
    starting_players.items()
)[:5]:
    print(
        player_id,
        info
    )

Starting players: 22
20055 {'player_name': 'Marc-André ter Stegen', 'team': 'Barcelona', 'start_minute': 0.0, 'end_minute': None}
43728 {'player_name': 'Óscar Mingueza García', 'team': 'Barcelona', 'start_minute': 0.0, 'end_minute': None}
5213 {'player_name': 'Gerard Piqué Bernabéu', 'team': 'Barcelona', 'start_minute': 0.0, 'end_minute': None}
5492 {'player_name': 'Samuel Yves Umtiti', 'team': 'Barcelona', 'start_minute': 0.0, 'end_minute': None}
5211 {'player_name': 'Jordi Alba Ramos', 'team': 'Barcelona', 'start_minute': 0.0, 'end_minute': None}


In [25]:
def initialise_live_exposure(starting_players):
    exposure = {}

    for player_id, info in starting_players.items():
        exposure[player_id] = {
            "player_name": info["player_name"],
            "team": info["team"],
            "on_pitch": True,
            "start_minute": 0.0,
            "minutes_played": 0.0,
            "last_update_minute": 0.0
        }

    return exposure


def update_live_exposure(
    event,
    exposure,
    current_minute
):
    raw_event = event["raw_event"]
    event_type = event["type"]

    for player_id, state in exposure.items():
        if state["on_pitch"]:
            delta = (
                current_minute
                - state["last_update_minute"]
            )

            if delta > 0:
                state["minutes_played"] += delta

            state["last_update_minute"] = current_minute

    if event_type == "Substitution":
        outgoing_id = event.get("player_id")

        replacement = (
            raw_event
            .get("substitution", {})
            .get("replacement")
        )

        if outgoing_id in exposure:
            exposure[
                outgoing_id
            ]["on_pitch"] = False

        if replacement is not None:
            replacement_id = replacement["id"]
            replacement_name = replacement["name"]

            exposure[replacement_id] = {
                "player_name": replacement_name,
                "team": event.get("team"),
                "on_pitch": True,
                "start_minute": current_minute,
                "minutes_played": 0.0,
                "last_update_minute": current_minute
            }

    elif event_type == "Player Off":
        player_id = event.get("player_id")

        if player_id in exposure:
            exposure[
                player_id
            ]["on_pitch"] = False

    elif event_type == "Player On":
        player_id = event.get("player_id")

        if player_id is not None:
            if player_id not in exposure:
                exposure[player_id] = {
                    "player_name": event.get("player_name"),
                    "team": event.get("team"),
                    "on_pitch": True,
                    "start_minute": current_minute,
                    "minutes_played": 0.0,
                    "last_update_minute": current_minute
                }
            else:
                exposure[player_id]["on_pitch"] = True
                exposure[player_id][
                    "last_update_minute"
                ] = current_minute

In [26]:
live_exposure = initialise_live_exposure(
    starting_players
)

for event in replay_events[:500]:
    current_minute = (
        event["minute"]
        + event["second"] / 60
    )

    update_live_exposure(
        event,
        live_exposure,
        current_minute
    )

exposure_df = pd.DataFrame(
    live_exposure
).T.reset_index(
    names="player_id"
)

display(
    exposure_df[
        [
            "player_name",
            "team",
            "on_pitch",
            "minutes_played"
        ]
    ]
    .sort_values(
        "minutes_played",
        ascending=False
    )
    .head(25)
    .round(2)
)

,player_name,team,on_pitch,minutes_played
0,Marc-André ter Stegen,Barcelona,True,11.316667
1,Óscar Mingueza García,Barcelona,True,11.316667
20,Emiliano Ariel Rigoni,Elche,True,11.316667
19,Pere Milla Peña,Elche,True,11.316667
18,Omenuke Mfulu,Elche,True,11.316667
17,José Raúl Gutiérrez Parejo,Elche,True,11.316667
16,Johan Andrés Mojica Palacio,Elche,True,11.316667
15,Miguel Ángel Garrido Cifuentes,Elche,True,11.316667
14,José Manuel Sánchez Guillén,Elche,True,11.316667
13,Gonzalo Cacicedo Verdú,Elche,True,11.316667


In [27]:
live_state_df = pd.DataFrame(
    live_player_state
).T.reset_index(
    names="player_id"
)

exposure_minutes = {
    player_id: info["minutes_played"]
    for player_id, info in live_exposure.items()
}

live_state_df["minutes_played"] = (
    live_state_df["player_id"]
    .map(exposure_minutes)
)

rate_metrics = [
    "actions",
    "pass_attempts",
    "progressive_passes",
    "carries",
    "shots",
    "miscontrols",
    "shot_assists"
]

for metric in rate_metrics:

    live_state_df[
        f"current_{metric}_per90"
    ] = np.where(
        live_state_df["minutes_played"] > 0,

        live_state_df[metric]
        / live_state_df["minutes_played"]
        * 90,

        np.nan
    )

live_state_df[
    "current_pass_completion_rate"
] = np.where(
    live_state_df["pass_attempts"] > 0,

    live_state_df["completed_passes"]
    / live_state_df["pass_attempts"],

    np.nan
)

display(
    live_state_df[
        [
            "player_name",
            "team",
            "minutes_played",
            "current_pass_attempts_per90",
            "historical_pass_attempts_per90",
            "current_progressive_passes_per90",
            "historical_progressive_passes_per90"
        ]
    ]
    .sort_values(
        "current_pass_attempts_per90",
        ascending=False
    )
    .head(15)
    .round(2)
)

,player_name,team,minutes_played,current_pass_attempts_per90,historical_pass_attempts_per90,current_progressive_passes_per90,historical_progressive_passes_per90
7,Gerard Piqué Bernabéu,Barcelona,11.32,214.727541,86.739576,23.858616,6.45933
6,Samuel Yves Umtiti,Barcelona,11.32,135.198822,78.346028,0.0,1.958651
17,Jordi Alba Ramos,Barcelona,11.32,111.340206,84.894053,7.952872,10.150376
18,Óscar Mingueza García,Barcelona,11.32,111.340206,94.392736,15.905744,2.451759
10,Francisco António Machado Mota de Castro Trincão,Barcelona,11.32,95.434462,28.723404,7.952872,0.0
16,Pedro González López,Barcelona,11.32,79.528719,80.952381,0.0,5.952381
9,Miralem Pjanić,Barcelona,11.32,63.622975,118.649518,0.0,0.0
12,Lionel Andrés Messi Cuccittini,Barcelona,11.32,55.670103,89.507861,23.858616,22.146275
8,Frenkie de Jong,Barcelona,11.32,47.717231,74.743677,0.0,5.536569
3,José Manuel Sánchez Guillén,Elche,11.32,47.717231,NaN,7.952872,NaN


In [28]:
def create_live_assessment(live_state_df):
    assessment = live_state_df.copy()

    assessment["pass_attempt_deviation"] = (
        assessment["current_pass_attempts_per90"]
        - assessment["historical_pass_attempts_per90"]
    )

    assessment["progression_deviation"] = (
        assessment["current_progressive_passes_per90"]
        - assessment["historical_progressive_passes_per90"]
    )

    assessment["carry_deviation"] = (
        assessment["current_carries_per90"]
        - assessment["historical_carries_per90"]
    )

    assessment["xpass_performance"] = (
        assessment["xpass_actual_completions"]
        - assessment["expected_completions"]
    )

    assessment["xpass_coverage"] = np.where(
        assessment["pass_attempts"] > 0,
        assessment["xpass_evaluated_passes"]
        / assessment["pass_attempts"],
        np.nan
    )

    assessment["enough_minutes"] = (
        assessment["minutes_played"] >= 10
    )

    assessment["enough_xpass"] = (
        assessment["xpass_evaluated_passes"] >= 5
    )

    assessment["has_historical_baseline"] = (
        assessment["previous_matches_available"] > 0
    )

    return assessment


live_assessment = create_live_assessment(
    live_state_df
)

display(
    live_assessment[
        [
            "player_name",
            "team",
            "minutes_played",
            "pass_attempt_deviation",
            "progression_deviation",
            "xpass_performance",
            "xpass_evaluated_passes",
            "xpass_coverage",
            "enough_minutes",
            "enough_xpass",
            "has_historical_baseline"
        ]
    ]
    .sort_values(
        "xpass_performance"
    )
    .round(2)
)

,player_name,team,minutes_played,pass_attempt_deviation,progression_deviation,xpass_performance,xpass_evaluated_passes,xpass_coverage,enough_minutes,enough_xpass,has_historical_baseline
0,Pere Milla Peña,Elche,11.32,NaN,NaN,-0.964257,2,1.0,True,False,False
21,Miguel Ángel Garrido Cifuentes,Elche,11.32,NaN,NaN,-0.79849,1,1.0,True,False,False
3,José Manuel Sánchez Guillén,Elche,11.32,NaN,NaN,-0.785794,6,1.0,True,True,False
15,Martin Braithwaite Christensen,Barcelona,11.32,6.401719,15.905744,-0.629303,3,1.0,True,False,True
13,Edgar Badía Guardiola,Elche,11.32,NaN,NaN,-0.482045,4,1.0,True,False,False
7,Gerard Piqué Bernabéu,Barcelona,11.32,127.987964,17.399285,-0.241926,26,0.962963,True,True,True
16,Pedro González López,Barcelona,11.32,-1.423662,-5.952381,-0.218069,10,1.0,True,True,True
20,Omenuke Mfulu,Elche,11.32,-19.96441,-3.988183,0.005377,1,1.0,True,False,True
5,Marc-André ter Stegen,Barcelona,11.32,7.81969,-0.922761,0.010075,3,0.75,True,False,True
17,Jordi Alba Ramos,Barcelona,11.32,26.446153,-2.197504,0.207328,13,0.928571,True,True,True


In [29]:
def generate_live_alerts(assessment):
    alerts = []

    for _, row in assessment.iterrows():
        player = row["player_name"]

        if not row["enough_minutes"]:
            continue

        # Contextual passing assessment
        if (
            row["enough_xpass"]
            and row["xpass_coverage"] >= 0.8
        ):
            xpass_difference = row["xpass_performance"]

            if xpass_difference <= -2.0:
                alerts.append({
                    "player": player,
                    "category": "Passing Execution",
                    "severity": "High",
                    "message": (
                        f"{player} is {abs(xpass_difference):.2f} "
                        "completed passes below contextual expectation."
                    )
                })

            elif xpass_difference <= -1.0:
                alerts.append({
                    "player": player,
                    "category": "Passing Execution",
                    "severity": "Moderate",
                    "message": (
                        f"{player} is {abs(xpass_difference):.2f} "
                        "completed passes below contextual expectation."
                    )
                })

        # Historical comparisons
        if not row["has_historical_baseline"]:
            continue

        historical_passes = row[
            "historical_pass_attempts_per90"
        ]

        current_passes = row[
            "current_pass_attempts_per90"
        ]

        if (
            pd.notna(historical_passes)
            and historical_passes > 0
        ):
            involvement_change = (
                (current_passes - historical_passes)
                / historical_passes
            )

            if involvement_change <= -0.40:
                alerts.append({
                    "player": player,
                    "category": "Involvement",
                    "severity": "Moderate",
                    "message": (
                        f"{player}'s passing involvement is "
                        f"{abs(involvement_change) * 100:.0f}% "
                        "below the historical baseline."
                    )
                })

        historical_progression = row[
            "historical_progressive_passes_per90"
        ]

        current_progression = row[
            "current_progressive_passes_per90"
        ]

        if (
            pd.notna(historical_progression)
            and historical_progression >= 2
        ):
            progression_change = (
                (current_progression - historical_progression)
                / historical_progression
            )

            if progression_change <= -0.50:
                alerts.append({
                    "player": player,
                    "category": "Progression",
                    "severity": "Moderate",
                    "message": (
                        f"{player}'s progressive passing rate is "
                        f"{abs(progression_change) * 100:.0f}% "
                        "below the historical baseline."
                    )
                })

    severity_order = {
        "Critical": 0,
        "High": 1,
        "Moderate": 2,
        "Informational": 3
    }

    alerts = sorted(
        alerts,
        key=lambda x: severity_order[x["severity"]]
    )

    return alerts

In [30]:
live_alerts = generate_live_alerts(
    live_assessment
)

print(
    "Alerts generated:",
    len(live_alerts)
)

for alert in live_alerts:
    print(
        f"[{alert['severity']}] "
        f"{alert['category']} | "
        f"{alert['message']}"
    )

Alerts generated: 9
[Moderate] Progression | Frenkie de Jong's progressive passing rate is 100% below the historical baseline.
[Moderate] Involvement | Miralem Pjanić's passing involvement is 46% below the historical baseline.
[Moderate] Involvement | Johan Andrés Mojica Palacio's passing involvement is 63% below the historical baseline.
[Moderate] Progression | Johan Andrés Mojica Palacio's progressive passing rate is 100% below the historical baseline.
[Moderate] Involvement | Antonio Barragán Fernández's passing involvement is 69% below the historical baseline.
[Moderate] Progression | Antonio Barragán Fernández's progressive passing rate is 100% below the historical baseline.
[Moderate] Progression | Pedro González López's progressive passing rate is 100% below the historical baseline.
[Moderate] Involvement | Omenuke Mfulu's passing involvement is 72% below the historical baseline.
[Moderate] Progression | Omenuke Mfulu's progressive passing rate is 100% below the historical basel

In [31]:
alert_history = {}

def update_alert_persistence(
    current_alerts,
    current_minute,
    required_occurrences=2
):
    persistent_alerts = []

    current_keys = set()

    for alert in current_alerts:
        key = (
            alert["player"],
            alert["category"]
        )

        current_keys.add(key)

        if key not in alert_history:
            alert_history[key] = {
                "count": 1,
                "first_seen": current_minute,
                "last_seen": current_minute
            }
        else:
            alert_history[key]["count"] += 1
            alert_history[key]["last_seen"] = current_minute

        alert["persistence_count"] = (
            alert_history[key]["count"]
        )

        alert["first_seen"] = (
            alert_history[key]["first_seen"]
        )

        if (
            alert_history[key]["count"]
            >= required_occurrences
        ):
            persistent_alerts.append(alert)

    for key in list(alert_history.keys()):
        if key not in current_keys:
            del alert_history[key]

    return persistent_alerts

In [32]:
checkpoint_minutes = [
    10,
    15,
    20,
    25,
    30,
    35,
    40,
    45
]

print(
    "Assessment checkpoints:",
    checkpoint_minutes
)

Assessment checkpoints: [10, 15, 20, 25, 30, 35, 40, 45]


In [33]:
def build_live_state_dataframe(
    live_player_state,
    live_exposure
):
    df = pd.DataFrame(
        live_player_state
    ).T.reset_index(
        names="player_id"
    )

    exposure_minutes = {
        player_id: info["minutes_played"]
        for player_id, info in live_exposure.items()
    }

    df["minutes_played"] = (
        df["player_id"]
        .map(exposure_minutes)
    )

    rate_metrics = [
        "actions",
        "pass_attempts",
        "progressive_passes",
        "carries",
        "shots",
        "miscontrols",
        "shot_assists"
    ]

    for metric in rate_metrics:
        df[
            f"current_{metric}_per90"
        ] = np.where(
            df["minutes_played"] > 0,
            df[metric]
            / df["minutes_played"]
            * 90,
            np.nan
        )

    df[
        "current_pass_completion_rate"
    ] = np.where(
        df["pass_attempts"] > 0,
        df["completed_passes"]
        / df["pass_attempts"],
        np.nan
    )

    return df

In [34]:
live_player_state = defaultdict(
    create_player_state
)

live_exposure = initialise_live_exposure(
    starting_players
)

alert_history = {}

checkpoint_results = []

next_checkpoint_index = 0

for event in replay_events:
    current_minute = (
        event["minute"]
        + event["second"] / 60
    )

    if current_minute > 45:
        break

    process_live_event(
        event,
        live_player_state
    )

    update_live_exposure(
        event,
        live_exposure,
        current_minute
    )

    while (
        next_checkpoint_index
        < len(checkpoint_minutes)
        and current_minute
        >= checkpoint_minutes[
            next_checkpoint_index
        ]
    ):
        checkpoint = checkpoint_minutes[
            next_checkpoint_index
        ]

        live_df = build_live_state_dataframe(
            live_player_state,
            live_exposure
        )

        # Reattach historical baseline values
        for player_id, state in live_player_state.items():
            history = history_lookup.get(
                player_id
            )

            if history is not None:
                state[
                    "previous_matches_available"
                ] = history[
                    "previous_matches_available"
                ]

                state[
                    "historical_pass_attempts_per90"
                ] = history[
                    "previous_reliable_pass_attempts_per90"
                ]

                state[
                    "historical_progressive_passes_per90"
                ] = history[
                    "previous_reliable_progressive_passes_per90"
                ]

                state[
                    "historical_carries_per90"
                ] = history[
                    "previous_reliable_carries_per90"
                ]

            else:
                state[
                    "previous_matches_available"
                ] = 0

                state[
                    "historical_pass_attempts_per90"
                ] = np.nan

                state[
                    "historical_progressive_passes_per90"
                ] = np.nan

                state[
                    "historical_carries_per90"
                ] = np.nan

        live_df = build_live_state_dataframe(
            live_player_state,
            live_exposure
        )

        assessment = create_live_assessment(
            live_df
        )

        raw_alerts = generate_live_alerts(
            assessment
        )

        persistent_alerts = (
            update_alert_persistence(
                raw_alerts,
                checkpoint,
                required_occurrences=2
            )
        )

        checkpoint_results.append({
            "checkpoint": checkpoint,
            "raw_alerts": len(raw_alerts),
            "persistent_alerts":
                persistent_alerts
        })

        next_checkpoint_index += 1

In [35]:
for result in checkpoint_results:
    print(
        f"\n--- {result['checkpoint']} minutes ---"
    )

    print(
        "Raw alerts:",
        result["raw_alerts"]
    )

    print(
        "Persistent alerts:",
        len(
            result["persistent_alerts"]
        )
    )

    for alert in result[
        "persistent_alerts"
    ]:
        print(
            f"[{alert['severity']}] "
            f"{alert['category']} | "
            f"{alert['message']} "
            f"(persistence="
            f"{alert['persistence_count']})"
        )


--- 10 minutes ---
Raw alerts: 10
Persistent alerts: 0

--- 15 minutes ---
Raw alerts: 6
Persistent alerts: 5
[Moderate] Involvement | Johan Andrés Mojica Palacio's passing involvement is 44% below the historical baseline. (persistence=2)
[Moderate] Involvement | Antonio Barragán Fernández's passing involvement is 65% below the historical baseline. (persistence=2)
[Moderate] Progression | Antonio Barragán Fernández's progressive passing rate is 100% below the historical baseline. (persistence=2)
[Moderate] Progression | Pedro González López's progressive passing rate is 100% below the historical baseline. (persistence=2)
[Moderate] Progression | Omenuke Mfulu's progressive passing rate is 100% below the historical baseline. (persistence=2)

--- 20 minutes ---
Raw alerts: 7
Persistent alerts: 5
[High] Passing Execution | Martin Braithwaite Christensen is 2.14 completed passes below contextual expectation. (persistence=2)
[Moderate] Involvement | Antonio Barragán Fernández's passing inv

In [36]:
popup_registry = {}


severity_rank = {
    "Informational": 1,
    "Moderate": 2,
    "High": 3,
    "Critical": 4
}


def create_popup_events(
    persistent_alerts,
    checkpoint
):
    popup_events = []

    active_keys = set()

    for alert in persistent_alerts:

        key = (
            alert["player"],
            alert["category"]
        )

        active_keys.add(key)

        severity = alert["severity"]

        if key not in popup_registry:

            popup_registry[key] = {
                "severity": severity,
                "first_popup": checkpoint,
                "last_seen": checkpoint,
                "active": True
            }

            popup = alert.copy()

            popup["popup_reason"] = (
                "New persistent alert"
            )

            popup["checkpoint"] = checkpoint

            popup_events.append(
                popup
            )

        else:

            previous_severity = (
                popup_registry[key][
                    "severity"
                ]
            )

            popup_registry[key][
                "last_seen"
            ] = checkpoint

            popup_registry[key][
                "active"
            ] = True

            if (
                severity_rank[severity]
                >
                severity_rank[
                    previous_severity
                ]
            ):

                popup_registry[key][
                    "severity"
                ] = severity

                popup = alert.copy()

                popup["popup_reason"] = (
                    "Severity increased"
                )

                popup["checkpoint"] = (
                    checkpoint
                )

                popup_events.append(
                    popup
                )

    for key in popup_registry:

        if key not in active_keys:

            popup_registry[key][
                "active"
            ] = False

    return popup_events

In [37]:
popup_registry = {}

all_popup_events = []

for result in checkpoint_results:

    checkpoint = result[
        "checkpoint"
    ]

    persistent_alerts = result[
        "persistent_alerts"
    ]

    popup_events = (
        create_popup_events(
            persistent_alerts,
            checkpoint
        )
    )

    all_popup_events.extend(
        popup_events
    )


popup_df = pd.DataFrame(
    all_popup_events
)

display(
    popup_df[
        [
            "checkpoint",
            "player",
            "category",
            "severity",
            "popup_reason",
            "message"
        ]
    ]
)

,checkpoint,player,category,severity,popup_reason,message
0,15,Johan Andrés Mojica Palacio,Involvement,Moderate,New persistent alert,Johan Andrés Mojica Palacio's passing involvem...
1,15,Antonio Barragán Fernández,Involvement,Moderate,New persistent alert,Antonio Barragán Fernández's passing involveme...
2,15,Antonio Barragán Fernández,Progression,Moderate,New persistent alert,Antonio Barragán Fernández's progressive passi...
3,15,Pedro González López,Progression,Moderate,New persistent alert,Pedro González López's progressive passing rat...
4,15,Omenuke Mfulu,Progression,Moderate,New persistent alert,Omenuke Mfulu's progressive passing rate is 10...
5,20,Martin Braithwaite Christensen,Passing Execution,High,New persistent alert,Martin Braithwaite Christensen is 2.14 complet...
6,25,José Manuel Sánchez Guillén,Passing Execution,Moderate,New persistent alert,José Manuel Sánchez Guillén is 1.69 completed ...
7,25,Emiliano Ariel Rigoni,Passing Execution,Moderate,New persistent alert,Emiliano Ariel Rigoni is 1.89 completed passes...
8,30,José Manuel Sánchez Guillén,Passing Execution,High,Severity increased,José Manuel Sánchez Guillén is 2.02 completed ...
9,30,Omenuke Mfulu,Passing Execution,High,New persistent alert,Omenuke Mfulu is 2.15 completed passes below c...


In [38]:
first_half_exposure = initialise_live_exposure(
    starting_players
)

first_half_last_minute = 0.0

for event in replay_events:

    if event["period"] != 1:
        continue

    current_minute = (
        event["minute"]
        + event["second"] / 60
    )

    update_live_exposure(
        event,
        first_half_exposure,
        current_minute
    )

    first_half_last_minute = max(
        first_half_last_minute,
        current_minute
    )


first_half_exposure_df = pd.DataFrame(
    first_half_exposure
).T.reset_index(
    names="player_id"
)

print(
    "First-half duration:",
    round(first_half_last_minute, 2),
    "minutes"
)

print(
    "Players tracked:",
    len(first_half_exposure_df)
)

display(
    first_half_exposure_df[
        [
            "player_name",
            "team",
            "on_pitch",
            "start_minute",
            "minutes_played"
        ]
    ]
    .sort_values(
        ["team", "minutes_played"],
        ascending=[True, False]
    )
    .round(2)
)

First-half duration: 45.95 minutes
Players tracked: 22


,player_name,team,on_pitch,start_minute,minutes_played
0,Marc-André ter Stegen,Barcelona,True,0.0,45.95
1,Óscar Mingueza García,Barcelona,True,0.0,45.95
2,Gerard Piqué Bernabéu,Barcelona,True,0.0,45.95
3,Samuel Yves Umtiti,Barcelona,True,0.0,45.95
4,Jordi Alba Ramos,Barcelona,True,0.0,45.95
5,Miralem Pjanić,Barcelona,True,0.0,45.95
6,Frenkie de Jong,Barcelona,True,0.0,45.95
7,Pedro González López,Barcelona,True,0.0,45.95
8,Francisco António Machado Mota de Castro Trincão,Barcelona,True,0.0,45.95
9,Martin Braithwaite Christensen,Barcelona,True,0.0,45.95


In [39]:
def close_exposure_at_checkpoint(
    exposure,
    checkpoint_minute
):
    for player_id, state in exposure.items():

        if state["on_pitch"]:

            delta = (
                checkpoint_minute
                - state["last_update_minute"]
            )

            if delta > 0:
                state["minutes_played"] += delta

            state["last_update_minute"] = (
                checkpoint_minute
            )

    return exposure

In [40]:
first_half_exposure = (
    close_exposure_at_checkpoint(
        first_half_exposure,
        first_half_last_minute
    )
)

first_half_exposure_df = pd.DataFrame(
    first_half_exposure
).T.reset_index(
    names="player_id"
)

display(
    first_half_exposure_df[
        [
            "player_name",
            "team",
            "on_pitch",
            "start_minute",
            "minutes_played"
        ]
    ]
    .sort_values(
        ["team", "minutes_played"],
        ascending=[True, False]
    )
    .round(2)
)

,player_name,team,on_pitch,start_minute,minutes_played
0,Marc-André ter Stegen,Barcelona,True,0.0,45.95
1,Óscar Mingueza García,Barcelona,True,0.0,45.95
2,Gerard Piqué Bernabéu,Barcelona,True,0.0,45.95
3,Samuel Yves Umtiti,Barcelona,True,0.0,45.95
4,Jordi Alba Ramos,Barcelona,True,0.0,45.95
5,Miralem Pjanić,Barcelona,True,0.0,45.95
6,Frenkie de Jong,Barcelona,True,0.0,45.95
7,Pedro González López,Barcelona,True,0.0,45.95
8,Francisco António Machado Mota de Castro Trincão,Barcelona,True,0.0,45.95
9,Martin Braithwaite Christensen,Barcelona,True,0.0,45.95


In [41]:
print(
    "Stored first_half_last_minute:",
    first_half_last_minute
)

period_1_times = [
    event["minute"] + event["second"] / 60
    for event in replay_events
    if event["period"] == 1
]

print(
    "Maximum Period 1 event time:",
    max(period_1_times)
)

print(
    "Last 15 Period 1 events:"
)

period_1_events = [
    event
    for event in replay_events
    if event["period"] == 1
]

for event in period_1_events[-15:]:
    print(
        event["index"],
        event["timestamp"],
        event["minute"],
        event["second"],
        event["type"],
        event["player_name"]
    )

Stored first_half_last_minute: 45.95
Maximum Period 1 event time: 45.95
Last 15 Period 1 events:
1981 00:45:04.661 45 4 Pass Francisco António Machado Mota de Castro Trincão
1982 00:45:05.726 45 5 Ball Receipt* Lionel Andrés Messi Cuccittini
1983 00:45:05.726 45 5 Clearance Antonio Barragán Fernández
1984 00:45:32.617 45 32 Pass Pedro González López
1985 00:45:33.832 45 33 Ball Receipt* Lionel Andrés Messi Cuccittini
1986 00:45:33.832 45 33 Carry Lionel Andrés Messi Cuccittini
1987 00:45:34.988 45 34 Pressure Miguel Ángel Garrido Cifuentes
1988 00:45:38.281 45 38 Pass Lionel Andrés Messi Cuccittini
1989 00:45:39.210 45 39 Ball Receipt* Pedro González López
1990 00:45:39.210 45 39 Carry Pedro González López
1991 00:45:40.273 45 40 Pass Pedro González López
1992 00:45:41.471 45 41 Ball Receipt* Gerard Piqué Bernabéu
1993 00:45:41.471 45 41 Goal Keeper Edgar Badía Guardiola
1994 00:45:57.650 45 57 Half End None
1995 00:45:57.650 45 57 Half End None


In [42]:
mojica_ids = [
    player_id
    for player_id, info
    in starting_players.items()
    if "Mojica" in info["player_name"]
]

print("Mojica ID:", mojica_ids)

for mojica_id in mojica_ids:

    for event in replay_events:

        if event["player_id"] != mojica_id:
            continue

        if event["type"] in [
            "Player Off",
            "Player On",
            "Substitution",
            "Bad Behaviour",
            "Foul Committed"
        ]:
            print(
                event["index"],
                event["period"],
                event["minute"],
                event["second"],
                event["type"],
                event["player_name"]
            )

Mojica ID: [5691]
1613 1 37 2 Player Off Johan Andrés Mojica Palacio
1622 1 37 35 Player On Johan Andrés Mojica Palacio


In [43]:
live_player_state = defaultdict(
    create_player_state
)

first_half_exposure = initialise_live_exposure(
    starting_players
)

first_half_last_minute = 0.0

for event in replay_events:

    if event["period"] != 1:
        continue

    current_minute = (
        event["minute"]
        + event["second"] / 60
    )

    process_live_event(
        event,
        live_player_state
    )

    update_live_exposure(
        event,
        first_half_exposure,
        current_minute
    )

    first_half_last_minute = max(
        first_half_last_minute,
        current_minute
    )


first_half_exposure = (
    close_exposure_at_checkpoint(
        first_half_exposure,
        first_half_last_minute
    )
)

In [44]:
for player_id, state in live_player_state.items():

    history = history_lookup.get(
        player_id
    )

    if history is not None:

        state[
            "previous_matches_available"
        ] = history[
            "previous_matches_available"
        ]

        state[
            "historical_pass_attempts_per90"
        ] = history[
            "previous_reliable_pass_attempts_per90"
        ]

        state[
            "historical_progressive_passes_per90"
        ] = history[
            "previous_reliable_progressive_passes_per90"
        ]

        state[
            "historical_carries_per90"
        ] = history[
            "previous_reliable_carries_per90"
        ]

        state[
            "historical_shots_per90"
        ] = history[
            "previous_reliable_shots_per90"
        ]

        state[
            "historical_miscontrols_per90"
        ] = history[
            "previous_reliable_miscontrols_per90"
        ]

        state[
            "historical_shot_assists_per90"
        ] = history[
            "previous_reliable_shot_assists_per90"
        ]

    else:

        state[
            "previous_matches_available"
        ] = 0

In [45]:
halftime_state = build_live_state_dataframe(
    live_player_state,
    first_half_exposure
)

print(
    "Players in halftime state:",
    len(halftime_state)
)

display(
    halftime_state[
        [
            "player_name",
            "team",
            "minutes_played",
            "actions",
            "pass_attempts",
            "completed_passes",
            "progressive_passes",
            "shots",
            "statsbomb_xg",
            "carries",
            "miscontrols",
            "xpass_evaluated_passes",
            "expected_completions"
        ]
    ]
    .sort_values(
        ["team", "actions"],
        ascending=[True, False]
    )
    .round(2)
)

Players in halftime state: 22


,player_name,team,minutes_played,actions,pass_attempts,completed_passes,progressive_passes,shots,statsbomb_xg,carries,miscontrols,xpass_evaluated_passes,expected_completions
7,Gerard Piqué Bernabéu,Barcelona,45.95,171,56,51,4,0,0.0,49,0,54,48.919365
9,Miralem Pjanić,Barcelona,45.95,154,50,44,7,0,0.0,43,0,49,42.720016
6,Samuel Yves Umtiti,Barcelona,45.95,137,50,47,3,0,0.0,38,0,50,47.201168
18,Óscar Mingueza García,Barcelona,45.95,136,42,41,5,0,0.0,37,0,42,37.494503
17,Jordi Alba Ramos,Barcelona,45.95,131,45,42,4,0,0.0,34,2,43,39.503868
8,Frenkie de Jong,Barcelona,45.95,117,34,32,3,0,0.0,33,0,34,30.78558
10,Francisco António Machado Mota de Castro Trincão,Barcelona,45.95,112,30,28,3,2,0.57441,30,0,30,27.359318
12,Lionel Andrés Messi Cuccittini,Barcelona,45.95,102,31,26,11,0,0.0,28,0,31,24.320137
16,Pedro González López,Barcelona,45.95,100,30,26,2,0,0.0,28,0,30,26.692987
15,Martin Braithwaite Christensen,Barcelona,45.95,53,11,7,2,0,0.0,13,1,11,9.021888


In [46]:
for target, model in forecasting_models.items():

    print("\nTARGET:", target)
    print("MODEL:", type(model).__name__)

    if hasattr(model, "feature_names_in_"):
        print(
            "Number of features:",
            len(model.feature_names_in_)
        )

        for feature in model.feature_names_in_:
            print(feature)

    else:
        print(
            "No feature_names_in_ found."
        )


TARGET: second_half_actions_per90
MODEL: Pipeline
Number of features: 37
previous_matches_available
previous_reliable_pass_attempts_per90
previous_reliable_progressive_passes_per90
previous_reliable_carries_per90
previous_reliable_shots_per90
previous_reliable_miscontrols_per90
previous_reliable_shot_assists_per90
last_3_reliable_pass_attempts_per90
last_3_reliable_progressive_passes_per90
last_3_reliable_carries_per90
last_3_reliable_shots_per90
last_3_reliable_miscontrols_per90
last_3_reliable_shot_assists_per90
first_half_actions_per90
first_half_pass_attempts_per90
first_half_completed_passes_per90
first_half_progressive_passes_per90
first_half_pass_completion_rate
first_half_shots_per90
first_half_statsbomb_xg_per90
first_half_carries_per90
first_half_miscontrols_per90
first_half_interceptions_per90
first_half_ball_recoveries_per90
first_half_dribbles_per90
first_half_duels_per90
first_half_pressures_per90
first_half_shot_assists_per90
first_half_mean_xpass
first_half_completions

In [48]:
import math

In [49]:
def create_trajectory_state():
    return {
        "actions": 0,
        "pass_attempts": 0,
        "completed_passes": 0,
        "progressive_passes": 0,
        "carries": 0,
        "miscontrols": 0,
        "pressures": 0
    }


early_state = defaultdict(
    create_trajectory_state
)

recent_state = defaultdict(
    create_trajectory_state
)


for event in replay_events:

    if event["period"] != 1:
        continue

    player_id = event["player_id"]

    if player_id is None:
        continue

    event_minute = (
        event["minute"]
        + event["second"] / 60
    )

    if event_minute < 30:
        state = early_state[player_id]
    else:
        state = recent_state[player_id]

    event_type = event["type"]
    raw_event = event["raw_event"]

    state["actions"] += 1

    if event_type == "Pass":

        state["pass_attempts"] += 1

        pass_data = raw_event.get(
            "pass",
            {}
        )

        if pass_data.get("outcome") is None:
            state["completed_passes"] += 1

        start = raw_event.get("location")
        end = pass_data.get("end_location")

        if (
            start is not None
            and end is not None
            and len(start) >= 2
            and len(end) >= 2
        ):
            start_distance = math.sqrt(
                (120 - start[0]) ** 2
                + (40 - start[1]) ** 2
            )

            end_distance = math.sqrt(
                (120 - end[0]) ** 2
                + (40 - end[1]) ** 2
            )

            if start_distance > 0:

                reduction = (
                    (
                        start_distance
                        - end_distance
                    )
                    / start_distance
                )

                if reduction >= 0.25:
                    state[
                        "progressive_passes"
                    ] += 1

    elif event_type == "Carry":
        state["carries"] += 1

    elif event_type == "Miscontrol":
        state["miscontrols"] += 1

    elif event_type == "Pressure":
        state["pressures"] += 1

In [50]:
early_exposure = initialise_live_exposure(
    starting_players
)

for event in replay_events:

    if event["period"] != 1:
        continue

    current_minute = (
        event["minute"]
        + event["second"] / 60
    )

    if current_minute > 30:
        break

    update_live_exposure(
        event,
        early_exposure,
        current_minute
    )


early_exposure = close_exposure_at_checkpoint(
    early_exposure,
    30.0
)


early_minutes_lookup = {
    player_id: state["minutes_played"]
    for player_id, state
    in early_exposure.items()
}


first_half_minutes_lookup = {
    player_id: state["minutes_played"]
    for player_id, state
    in first_half_exposure.items()
}


recent_minutes_lookup = {}

for player_id, first_half_minutes in (
    first_half_minutes_lookup.items()
):

    early_minutes = (
        early_minutes_lookup.get(
            player_id,
            0.0
        )
    )

    recent_minutes_lookup[player_id] = max(
        first_half_minutes
        - early_minutes,
        0.0
    )

In [51]:
trajectory_features = {}


for player_id in first_half_minutes_lookup:

    early = early_state[player_id]
    recent = recent_state[player_id]

    early_minutes = (
        early_minutes_lookup.get(
            player_id,
            0.0
        )
    )

    recent_minutes = (
        recent_minutes_lookup.get(
            player_id,
            0.0
        )
    )

    player_features = {}

    for metric in [
        "actions",
        "progressive_passes",
        "carries",
        "miscontrols",
        "pressures"
    ]:

        if early_minutes > 0:
            early_rate = (
                early[metric]
                / early_minutes
                * 90
            )
        else:
            early_rate = np.nan

        if recent_minutes > 0:
            recent_rate = (
                recent[metric]
                / recent_minutes
                * 90
            )
        else:
            recent_rate = np.nan

        player_features[
            f"{metric}_trajectory"
        ] = (
            recent_rate
            - early_rate
        )

    if early["pass_attempts"] > 0:
        early_completion = (
            early["completed_passes"]
            / early["pass_attempts"]
        )
    else:
        early_completion = np.nan

    if recent["pass_attempts"] > 0:
        recent_completion = (
            recent["completed_passes"]
            / recent["pass_attempts"]
        )
    else:
        recent_completion = np.nan

    player_features[
        "pass_completion_trajectory"
    ] = (
        recent_completion
        - early_completion
    )

    trajectory_features[
        player_id
    ] = player_features

In [52]:
trajectory_df = (
    pd.DataFrame(
        trajectory_features
    )
    .T
    .reset_index(
        names="player_id"
    )
)

trajectory_df["player_name"] = (
    trajectory_df["player_id"]
    .map({
        player_id: state["player_name"]
        for player_id, state
        in live_player_state.items()
    })
)

display(
    trajectory_df[
        [
            "player_name",
            "actions_trajectory",
            "progressive_passes_trajectory",
            "carries_trajectory",
            "miscontrols_trajectory",
            "pressures_trajectory",
            "pass_completion_trajectory"
        ]
    ]
    .round(3)
)

,player_name,actions_trajectory,progressive_passes_trajectory,carries_trajectory,miscontrols_trajectory,pressures_trajectory,pass_completion_trajectory
0,Marc-André ter Stegen,14.069,5.643,-6.357,0.000,0.000,0.000
1,Óscar Mingueza García,-62.295,2.285,-7.288,0.000,-9.357,0.033
2,Gerard Piqué Bernabéu,-132.724,-12.000,-51.931,0.000,-3.715,0.116
3,Samuel Yves Umtiti,-134.436,-0.357,-44.859,0.000,-6.000,-0.031
4,Jordi Alba Ramos,-55.937,-12.000,-15.574,11.285,19.571,0.086
5,Miralem Pjanić,21.987,-3.715,-8.003,0.000,5.285,-0.050
6,Frenkie de Jong,-39.865,-0.357,-21.216,0.000,-3.000,-0.038
7,Pedro González López,-49.364,2.643,-23.502,0.000,25.213,-0.127
8,Francisco António Machado Mota de Castro Trincão,-68.078,-0.357,-29.502,0.000,25.213,-0.099
9,Martin Braithwaite Christensen,-107.144,-6.000,-30.357,-3.000,-3.357,NaN


In [53]:
def build_halftime_forecast_features(
    halftime_state,
    first_half_exposure,
    history_lookup,
    trajectory_features
):
    rows = []

    for _, player in halftime_state.iterrows():

        player_id = player["player_id"]

        minutes = player["minutes_played"]

        if pd.isna(minutes) or minutes <= 0:
            continue

        history = history_lookup.get(
            player_id,
            {}
        )

        row = {
            "player_id": player_id,
            "player_name": player["player_name"],
            "team": player["team"],
            "first_half_minutes": minutes
        }

        history_features = [
            "previous_matches_available",

            "previous_reliable_pass_attempts_per90",
            "previous_reliable_progressive_passes_per90",
            "previous_reliable_carries_per90",
            "previous_reliable_shots_per90",
            "previous_reliable_miscontrols_per90",
            "previous_reliable_shot_assists_per90",

            "last_3_reliable_pass_attempts_per90",
            "last_3_reliable_progressive_passes_per90",
            "last_3_reliable_carries_per90",
            "last_3_reliable_shots_per90",
            "last_3_reliable_miscontrols_per90",
            "last_3_reliable_shot_assists_per90"
        ]

        for feature in history_features:
            row[feature] = history.get(
                feature,
                np.nan
            )

        def per90(value):
            return (
                value
                / minutes
                * 90
            )

        row["first_half_actions_per90"] = (
            per90(player["actions"])
        )

        row["first_half_pass_attempts_per90"] = (
            per90(player["pass_attempts"])
        )

        row["first_half_completed_passes_per90"] = (
            per90(player["completed_passes"])
        )

        row["first_half_progressive_passes_per90"] = (
            per90(player["progressive_passes"])
        )

        if player["pass_attempts"] > 0:
            row[
                "first_half_pass_completion_rate"
            ] = (
                player["completed_passes"]
                / player["pass_attempts"]
            )
        else:
            row[
                "first_half_pass_completion_rate"
            ] = np.nan

        row["first_half_shots_per90"] = (
            per90(player["shots"])
        )

        row["first_half_statsbomb_xg_per90"] = (
            per90(player["statsbomb_xg"])
        )

        row["first_half_carries_per90"] = (
            per90(player["carries"])
        )

        row["first_half_miscontrols_per90"] = (
            per90(player["miscontrols"])
        )

        row["first_half_interceptions_per90"] = (
            per90(player["interceptions"])
        )

        row["first_half_ball_recoveries_per90"] = (
            per90(player["ball_recoveries"])
        )

        row["first_half_dribbles_per90"] = (
            per90(player["dribbles"])
        )

        row["first_half_duels_per90"] = (
            per90(player["duels"])
        )

        row["first_half_pressures_per90"] = (
            per90(player["pressures"])
        )

        row["first_half_shot_assists_per90"] = (
            per90(player["shot_assists"])
        )

        evaluated = player[
            "xpass_evaluated_passes"
        ]

        expected = player[
            "expected_completions"
        ]

        actual_evaluated = player[
            "xpass_actual_completions"
        ]

        if evaluated > 0:

            mean_xpass = (
                expected / evaluated
            )

            actual_evaluated_rate = (
                actual_evaluated
                / evaluated
            )

            row[
                "first_half_mean_xpass"
            ] = mean_xpass

            row[
                "first_half_completions_above_expected"
            ] = (
                actual_evaluated
                - expected
            )

            row[
                "first_half_completion_rate_above_expected"
            ] = (
                actual_evaluated_rate
                - mean_xpass
            )

        else:

            row[
                "first_half_mean_xpass"
            ] = np.nan

            row[
                "first_half_completions_above_expected"
            ] = np.nan

            row[
                "first_half_completion_rate_above_expected"
            ] = np.nan

        trajectory = trajectory_features.get(
            player_id,
            {}
        )

        for feature in [
            "actions_trajectory",
            "progressive_passes_trajectory",
            "carries_trajectory",
            "miscontrols_trajectory",
            "pressures_trajectory",
            "pass_completion_trajectory"
        ]:
            row[feature] = trajectory.get(
                feature,
                np.nan
            )

        rows.append(row)

    return pd.DataFrame(rows)

In [54]:
halftime_features = (
    build_halftime_forecast_features(
        halftime_state,
        first_half_exposure,
        history_lookup,
        trajectory_features
    )
)

print(
    "Halftime feature table:",
    halftime_features.shape
)

print(
    "Players:",
    halftime_features["player_id"].nunique()
)

display(
    halftime_features[
        [
            "player_name",
            "team",
            "first_half_minutes",
            "previous_matches_available",
            "first_half_actions_per90",
            "first_half_pass_completion_rate",
            "first_half_mean_xpass",
            "first_half_completions_above_expected",
            "actions_trajectory"
        ]
    ]
    .round(3)
)

Halftime feature table: (22, 41)
Players: 22


,player_name,team,first_half_minutes,previous_matches_available,first_half_actions_per90,first_half_pass_completion_rate,first_half_mean_xpass,first_half_completions_above_expected,actions_trajectory
0,Pere Milla Peña,Elche,45.95,0,170.403,0.929,0.934,-0.070,15.564
1,José Raúl Gutiérrez Parejo,Elche,45.95,0,125.354,1.000,0.911,1.251,32.708
2,Gonzalo Cacicedo Verdú,Elche,45.95,0,160.609,1.000,0.817,4.392,82.420
3,José Manuel Sánchez Guillén,Elche,45.95,0,127.312,0.636,0.749,-3.225,-160.429
4,Lucas Ariel Boyé,Elche,45.95,1,154.733,0.533,0.707,-1.893,-38.219
5,Marc-André ter Stegen,Barcelona,45.95,153,52.884,1.000,0.993,0.059,14.069
6,Samuel Yves Umtiti,Barcelona,45.95,116,268.335,0.940,0.944,-0.201,-134.436
7,Gerard Piqué Bernabéu,Barcelona,45.95,331,334.929,0.911,0.906,0.081,-132.724
8,Frenkie de Jong,Barcelona,45.95,45,229.162,0.941,0.905,1.214,-39.865
9,Miralem Pjanić,Barcelona,45.95,49,301.632,0.880,0.872,0.280,21.987


In [55]:
for target, model in forecasting_models.items():

    required_features = list(
        model.feature_names_in_
    )

    missing_features = [
        feature
        for feature in required_features
        if feature not in halftime_features.columns
    ]

    print(
        target,
        "| required:",
        len(required_features),
        "| missing:",
        len(missing_features)
    )

    if missing_features:
        print(
            "Missing features:",
            missing_features
        )

second_half_actions_per90 | required: 37 | missing: 0
second_half_miscontrols_per90 | required: 28 | missing: 0
second_half_pass_completion_rate | required: 31 | missing: 0
second_half_progressive_passes_per90 | required: 28 | missing: 0
second_half_statsbomb_xg_per90 | required: 28 | missing: 0


In [56]:
halftime_forecasts = halftime_features[
    [
        "player_id",
        "player_name",
        "team",
        "first_half_minutes"
    ]
].copy()


for target, model in forecasting_models.items():

    required_features = list(
        model.feature_names_in_
    )

    X_live = halftime_features[
        required_features
    ].copy()

    X_live = X_live.apply(
        pd.to_numeric,
        errors="coerce"
    ).astype(float)

    halftime_forecasts[
        target
    ] = model.predict(
        X_live
    )


print(
    "Forecasts generated:",
    len(halftime_forecasts)
)

display(
    halftime_forecasts.round(3)
)

Forecasts generated: 22


,player_id,player_name,team,first_half_minutes,second_half_actions_per90,second_half_miscontrols_per90,second_half_pass_completion_rate,second_half_progressive_passes_per90,second_half_statsbomb_xg_per90
0,12072,Pere Milla Peña,Elche,45.95,153.242,1.796,0.828,3.198,0.146
1,24517,José Raúl Gutiérrez Parejo,Elche,45.95,141.262,0.787,0.861,3.621,0.091
2,24169,Gonzalo Cacicedo Verdú,Elche,45.95,154.546,0.245,0.838,7.207,0.033
3,9857,José Manuel Sánchez Guillén,Elche,45.95,124.933,0.680,0.749,8.167,0.043
4,7064,Lucas Ariel Boyé,Elche,45.95,147.383,2.305,0.683,4.430,0.150
5,20055,Marc-André ter Stegen,Barcelona,45.95,60.079,-0.270,0.896,4.851,0.040
6,5492,Samuel Yves Umtiti,Barcelona,45.95,209.669,-0.221,0.961,5.333,0.026
7,5213,Gerard Piqué Bernabéu,Barcelona,45.95,219.538,-0.127,0.971,6.057,0.090
8,8118,Frenkie de Jong,Barcelona,45.95,222.025,0.528,0.940,4.642,0.098
9,6947,Miralem Pjanić,Barcelona,45.95,265.614,0.868,0.952,6.884,0.171


In [57]:
forecast_display = (
    halftime_forecasts.rename(
        columns={
            "second_half_actions_per90":
                "Predicted Actions/90",

            "second_half_progressive_passes_per90":
                "Predicted Progressive Passes/90",

            "second_half_pass_completion_rate":
                "Predicted Pass Completion",

            "second_half_miscontrols_per90":
                "Predicted Miscontrols/90",

            "second_half_statsbomb_xg_per90":
                "Predicted xG/90"
        }
    )
)


forecast_display[
    "Predicted Pass Completion"
] = (
    forecast_display[
        "Predicted Pass Completion"
    ] * 100
)


display(
    forecast_display[
        [
            "player_name",
            "team",
            "Predicted Actions/90",
            "Predicted Progressive Passes/90",
            "Predicted Pass Completion",
            "Predicted Miscontrols/90",
            "Predicted xG/90"
        ]
    ]
    .sort_values(
        [
            "team",
            "Predicted Actions/90"
        ],
        ascending=[
            True,
            False
        ]
    )
    .round(2)
)

,player_name,team,Predicted Actions/90,Predicted Progressive Passes/90,Predicted Pass Completion,Predicted Miscontrols/90,Predicted xG/90
9,Miralem Pjanić,Barcelona,265.61,6.88,95.19,0.87,0.17
12,Lionel Andrés Messi Cuccittini,Barcelona,229.24,15.78,81.17,0.91,0.43
8,Frenkie de Jong,Barcelona,222.02,4.64,94.00,0.53,0.10
7,Gerard Piqué Bernabéu,Barcelona,219.54,6.06,97.11,-0.13,0.09
16,Pedro González López,Barcelona,219.32,5.30,89.57,0.98,0.06
17,Jordi Alba Ramos,Barcelona,213.02,8.87,89.52,0.26,0.04
6,Samuel Yves Umtiti,Barcelona,209.67,5.33,96.08,-0.22,0.03
18,Óscar Mingueza García,Barcelona,209.58,6.13,92.55,0.58,0.10
10,Francisco António Machado Mota de Castro Trincão,Barcelona,158.04,4.46,88.21,2.18,0.35
15,Martin Braithwaite Christensen,Barcelona,112.34,4.15,75.21,1.47,0.25


In [58]:
forecast_dashboard = halftime_forecasts.copy()

non_negative_targets = [
    "second_half_actions_per90",
    "second_half_progressive_passes_per90",
    "second_half_miscontrols_per90",
    "second_half_statsbomb_xg_per90"
]

for target in non_negative_targets:
    forecast_dashboard[target] = (
        forecast_dashboard[target]
        .clip(lower=0)
    )

forecast_dashboard[
    "second_half_pass_completion_rate"
] = (
    forecast_dashboard[
        "second_half_pass_completion_rate"
    ]
    .clip(
        lower=0,
        upper=1
    )
)

forecast_dashboard[
    "predicted_pass_completion_pct"
] = (
    forecast_dashboard[
        "second_half_pass_completion_rate"
    ] * 100
)

display(
    forecast_dashboard[
        [
            "player_name",
            "team",
            "second_half_actions_per90",
            "second_half_progressive_passes_per90",
            "predicted_pass_completion_pct",
            "second_half_miscontrols_per90",
            "second_half_statsbomb_xg_per90"
        ]
    ]
    .round(2)
)

,player_name,team,second_half_actions_per90,second_half_progressive_passes_per90,predicted_pass_completion_pct,second_half_miscontrols_per90,second_half_statsbomb_xg_per90
0,Pere Milla Peña,Elche,153.24,3.20,82.79,1.80,0.15
1,José Raúl Gutiérrez Parejo,Elche,141.26,3.62,86.14,0.79,0.09
2,Gonzalo Cacicedo Verdú,Elche,154.55,7.21,83.77,0.25,0.03
3,José Manuel Sánchez Guillén,Elche,124.93,8.17,74.94,0.68,0.04
4,Lucas Ariel Boyé,Elche,147.38,4.43,68.28,2.31,0.15
5,Marc-André ter Stegen,Barcelona,60.08,4.85,89.63,0.00,0.04
6,Samuel Yves Umtiti,Barcelona,209.67,5.33,96.08,0.00,0.03
7,Gerard Piqué Bernabéu,Barcelona,219.54,6.06,97.11,0.00,0.09
8,Frenkie de Jong,Barcelona,222.02,4.64,94.00,0.53,0.10
9,Miralem Pjanić,Barcelona,265.61,6.88,95.19,0.87,0.17


In [59]:
def build_decision_support(
    forecast_dashboard,
    halftime_features,
    persistent_alerts
):
    decisions = []

    active_alerts = {}

    for alert in persistent_alerts:
        player = alert["player"]

        if player not in active_alerts:
            active_alerts[player] = []

        active_alerts[player].append(
            alert
        )

    for _, forecast in forecast_dashboard.iterrows():

        player = forecast["player_name"]

        player_alerts = active_alerts.get(
            player,
            []
        )

        if not player_alerts:
            continue

        categories = {
            alert["category"]
            for alert in player_alerts
        }

        severities = {
            alert["severity"]
            for alert in player_alerts
        }

        reasons = []

        recommendation = "Monitor"

        if "Passing Execution" in categories:
            reasons.append(
                "first-half passing execution remained "
                "below contextual expectation"
            )

        if "Progression" in categories:
            reasons.append(
                "progressive passing remained below "
                "the player's historical baseline"
            )

        if "Involvement" in categories:
            reasons.append(
                "passing involvement remained below "
                "the player's historical baseline"
            )

        predicted_completion = forecast[
            "second_half_pass_completion_rate"
        ]

        predicted_progression = forecast[
            "second_half_progressive_passes_per90"
        ]

        predicted_miscontrols = forecast[
            "second_half_miscontrols_per90"
        ]

        if (
            "High" in severities
            and predicted_completion < 0.80
        ):
            recommendation = (
                "Consider role adjustment or additional support"
            )

            reasons.append(
                "second-half passing efficiency is "
                "forecast to remain relatively low"
            )

        elif (
            "Progression" in categories
            and predicted_progression < 5
        ):
            recommendation = (
                "Consider role adjustment or additional support"
            )

            reasons.append(
                "second-half progression is forecast "
                "to remain limited"
            )

        if (
            "High" in severities
            and predicted_miscontrols >= 2
        ):
            recommendation = (
                "Consider substitution monitoring"
            )

            reasons.append(
                "forecast indicates elevated "
                "second-half ball-security risk"
            )

        decisions.append({
            "player": player,
            "team": forecast["team"],
            "recommendation": recommendation,
            "reason": "; ".join(reasons),
            "predicted_actions_per90":
                forecast["second_half_actions_per90"],
            "predicted_progressive_passes_per90":
                predicted_progression,
            "predicted_pass_completion_pct":
                predicted_completion * 100,
            "predicted_miscontrols_per90":
                predicted_miscontrols,
            "predicted_xg_per90":
                forecast["second_half_statsbomb_xg_per90"]
        })

    return pd.DataFrame(decisions)

In [60]:
halftime_alerts = (
    checkpoint_results[-1][
        "persistent_alerts"
    ]
)

print(
    "Persistent alerts active at halftime:",
    len(halftime_alerts)
)

for alert in halftime_alerts:
    print(
        alert["player"],
        "|",
        alert["category"],
        "|",
        alert["severity"]
    )

Persistent alerts active at halftime: 8
José Manuel Sánchez Guillén | Passing Execution | High
Martin Braithwaite Christensen | Passing Execution | High
Omenuke Mfulu | Passing Execution | High
Lucas Ariel Boyé | Passing Execution | Moderate
Antonio Barragán Fernández | Involvement | Moderate
Antonio Barragán Fernández | Progression | Moderate
Pedro González López | Progression | Moderate
Omenuke Mfulu | Progression | Moderate


In [61]:
halftime_decisions = build_decision_support(
    forecast_dashboard,
    halftime_features,
    halftime_alerts
)

display(
    halftime_decisions.round(2)
)

,player,team,recommendation,reason,predicted_actions_per90,predicted_progressive_passes_per90,predicted_pass_completion_pct,predicted_miscontrols_per90,predicted_xg_per90
0,José Manuel Sánchez Guillén,Elche,Consider role adjustment or additional support,first-half passing execution remained below co...,124.93,8.17,74.94,0.68,0.04
1,Lucas Ariel Boyé,Elche,Monitor,first-half passing execution remained below co...,147.38,4.43,68.28,2.31,0.15
2,Antonio Barragán Fernández,Elche,Monitor,progressive passing remained below the player'...,124.85,5.08,85.24,0.35,0.02
3,Martin Braithwaite Christensen,Barcelona,Consider role adjustment or additional support,first-half passing execution remained below co...,112.34,4.15,75.21,1.47,0.25
4,Pedro González López,Barcelona,Monitor,progressive passing remained below the player'...,219.32,5.30,89.57,0.98,0.06
5,Omenuke Mfulu,Elche,Consider role adjustment or additional support,first-half passing execution remained below co...,154.64,5.26,78.57,1.55,0.09


In [62]:
def build_decision_support(
    forecast_dashboard,
    halftime_features,
    persistent_alerts
):
    decisions = []

    severity_rank = {
        "Informational": 1,
        "Moderate": 2,
        "High": 3,
        "Critical": 4
    }

    active_alerts = {}

    for alert in persistent_alerts:
        player = alert["player"]

        if player not in active_alerts:
            active_alerts[player] = []

        active_alerts[player].append(alert)

    for _, forecast in forecast_dashboard.iterrows():

        player = forecast["player_name"]

        player_alerts = active_alerts.get(
            player,
            []
        )

        if not player_alerts:
            continue

        categories = sorted({
            alert["category"]
            for alert in player_alerts
        })

        highest_severity = max(
            (
                alert["severity"]
                for alert in player_alerts
            ),
            key=lambda x: severity_rank[x]
        )

        reasons = []

        recommendation = "Monitor"

        if "Passing Execution" in categories:
            reasons.append(
                "first-half passing execution remained "
                "below contextual expectation"
            )

        if "Progression" in categories:
            reasons.append(
                "progressive passing remained below "
                "the player's historical baseline"
            )

        if "Involvement" in categories:
            reasons.append(
                "passing involvement remained below "
                "the player's historical baseline"
            )

        predicted_completion = forecast[
            "second_half_pass_completion_rate"
        ]

        predicted_progression = forecast[
            "second_half_progressive_passes_per90"
        ]

        predicted_miscontrols = forecast[
            "second_half_miscontrols_per90"
        ]

        if (
            highest_severity in ["High", "Critical"]
            and predicted_completion < 0.80
        ):
            recommendation = (
                "Consider role adjustment or additional support"
            )

            reasons.append(
                "second-half passing efficiency is "
                "forecast to remain relatively low"
            )

        elif (
            "Progression" in categories
            and predicted_progression < 5
        ):
            recommendation = (
                "Consider role adjustment or additional support"
            )

            reasons.append(
                "second-half progression is forecast "
                "to remain limited"
            )

        if (
            highest_severity in ["High", "Critical"]
            and predicted_miscontrols >= 2
        ):
            recommendation = (
                "Consider substitution monitoring"
            )

            reasons.append(
                "forecast indicates elevated "
                "second-half ball-security risk"
            )

        decisions.append({
            "player": player,
            "team": forecast["team"],
            "severity": highest_severity,
            "alert_categories": ", ".join(categories),
            "recommendation": recommendation,
            "reason": "; ".join(reasons),

            "predicted_actions_per90":
                forecast["second_half_actions_per90"],

            "predicted_progressive_passes_per90":
                predicted_progression,

            "predicted_pass_completion_pct":
                predicted_completion * 100,

            "predicted_miscontrols_per90":
                predicted_miscontrols,

            "predicted_xg_per90":
                forecast[
                    "second_half_statsbomb_xg_per90"
                ]
        })

    decisions_df = pd.DataFrame(
        decisions
    )

    if not decisions_df.empty:

        decisions_df["priority"] = (
            decisions_df["severity"]
            .map(severity_rank)
        )

        decisions_df = (
            decisions_df
            .sort_values(
                [
                    "priority",
                    "team",
                    "player"
                ],
                ascending=[
                    False,
                    True,
                    True
                ]
            )
            .drop(
                columns="priority"
            )
            .reset_index(
                drop=True
            )
        )

    return decisions_df

In [63]:
halftime_decisions = build_decision_support(
    forecast_dashboard,
    halftime_features,
    halftime_alerts
)

display(
    halftime_decisions[
        [
            "player",
            "team",
            "severity",
            "alert_categories",
            "recommendation",
            "predicted_pass_completion_pct",
            "predicted_progressive_passes_per90",
            "predicted_miscontrols_per90",
            "predicted_xg_per90"
        ]
    ]
    .round(2)
)

,player,team,severity,alert_categories,recommendation,predicted_pass_completion_pct,predicted_progressive_passes_per90,predicted_miscontrols_per90,predicted_xg_per90
0,Martin Braithwaite Christensen,Barcelona,High,Passing Execution,Consider role adjustment or additional support,75.21,4.15,1.47,0.25
1,José Manuel Sánchez Guillén,Elche,High,Passing Execution,Consider role adjustment or additional support,74.94,8.17,0.68,0.04
2,Omenuke Mfulu,Elche,High,"Passing Execution, Progression",Consider role adjustment or additional support,78.57,5.26,1.55,0.09
3,Pedro González López,Barcelona,Moderate,Progression,Monitor,89.57,5.30,0.98,0.06
4,Antonio Barragán Fernández,Elche,Moderate,"Involvement, Progression",Monitor,85.24,5.08,0.35,0.02
5,Lucas Ariel Boyé,Elche,Moderate,Passing Execution,Monitor,68.28,4.43,2.31,0.15


In [64]:
import importlib.util

packages = {
    "kafka": "Kafka Python",
    "pyspark": "PySpark"
}

for package, name in packages.items():
    installed = importlib.util.find_spec(package) is not None
    print(f"{name}: {'Installed' if installed else 'Not installed'}")

Kafka Python: Not installed
PySpark: Not installed


In [65]:
%pip install kafka-python pyspark

     ---------------------------------------- 0.0/450.1 MB ? eta -:--:--
     --------------------------------------- 1.8/450.1 MB 10.8 MB/s eta 0:00:42
     --------------------------------------- 4.2/450.1 MB 10.9 MB/s eta 0:00:42
      -------------------------------------- 6.8/450.1 MB 11.2 MB/s eta 0:00:40
      -------------------------------------- 9.2/450.1 MB 11.2 MB/s eta 0:00:40
      ------------------------------------- 11.0/450.1 MB 10.8 MB/s eta 0:00:41
     - ------------------------------------ 13.1/450.1 MB 10.7 MB/s eta 0:00:41
     - ------------------------------------ 15.7/450.1 MB 10.8 MB/s eta 0:00:41
     - ------------------------------------ 18.1/450.1 MB 10.9 MB/s eta 0:00:40
     - ------------------------------------ 20.2/450.1 MB 10.9 MB/s eta 0:00:40
     - ------------------------------------ 21.8/450.1 MB 10.5 MB/s eta 0:00:41
     -- ----------------------------------- 24.4/450.1 MB 10.7 MB/s eta 0:00:40
     -- ----------------------------------- 25.


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [66]:
import importlib.util

packages = {
    "kafka": "Kafka Python",
    "pyspark": "PySpark"
}

for package, name in packages.items():
    installed = importlib.util.find_spec(package) is not None
    print(f"{name}: {'Installed' if installed else 'Not installed'}")

Kafka Python: Installed
PySpark: Installed


In [1]:
from kafka import KafkaProducer
import json

producer = KafkaProducer(
    bootstrap_servers="localhost:9092",
    value_serializer=lambda value: json.dumps(value).encode("utf-8")
)

print("Kafka producer connected.")

Kafka producer connected.


In [3]:
import json

match_id = 3764440

events_path = f"open-data-master/data/events/{match_id}.json"
three_sixty_path = f"open-data-master/data/three-sixty/{match_id}.json"

with open(events_path, "r", encoding="utf-8") as f:
    events = json.load(f)

with open(three_sixty_path, "r", encoding="utf-8") as f:
    three_sixty = json.load(f)

three_sixty_lookup = {
    item["event_uuid"]: item
    for item in three_sixty
}

replay_events = []

for event in events:

    event_id = event["id"]

    spatial = three_sixty_lookup.get(
        event_id,
        {}
    )

    player = event.get("player")

    replay_events.append({
        "event_id": event_id,
        "index": event.get("index"),
        "period": event.get("period"),
        "timestamp": event.get("timestamp"),
        "minute": event.get("minute"),
        "second": event.get("second"),
        "type": event.get("type", {}).get("name"),
        "team": event.get("team", {}).get("name"),
        "player_id": (
            player.get("id")
            if player
            else None
        ),
        "player_name": (
            player.get("name")
            if player
            else None
        ),
        "raw_event": event,
        "freeze_frame": spatial.get("freeze_frame"),
        "visible_area": spatial.get("visible_area")
    })

replay_events = sorted(
    replay_events,
    key=lambda x: x["index"]
)

print("Replay events:", len(replay_events))
print("360 records:", len(three_sixty_lookup))

Replay events: 4160
360 records: 3914


In [4]:
from kafka import KafkaProducer

producer = KafkaProducer(
    bootstrap_servers="localhost:9092",
    value_serializer=lambda value: json.dumps(value).encode("utf-8")
)

test_event = replay_events[4]

future = producer.send(
    "football-events",
    value=test_event
)

metadata = future.get(timeout=10)

producer.flush()

print("Event sent successfully")
print("Topic:", metadata.topic)
print("Partition:", metadata.partition)
print("Offset:", metadata.offset)

print(
    "Event:",
    test_event["type"],
    "| Team:",
    test_event["team"],
    "| Player:",
    test_event["player_name"]
)

Event sent successfully
Topic: football-events
Partition: 0
Offset: 0
Event: Pass | Team: Elche | Player: Pere Milla Peña


In [5]:
from kafka import KafkaConsumer
import json

consumer = KafkaConsumer(
    "football-events",
    bootstrap_servers="localhost:9092",
    auto_offset_reset="earliest",
    enable_auto_commit=False,
    group_id="football-test-consumer",
    value_deserializer=lambda value: json.loads(
        value.decode("utf-8")
    ),
    consumer_timeout_ms=5000
)

for message in consumer:

    event = message.value

    print("Event received successfully")
    print("Topic:", message.topic)
    print("Partition:", message.partition)
    print("Offset:", message.offset)

    print(
        "Event:",
        event["type"],
        "| Team:",
        event["team"],
        "| Player:",
        event["player_name"]
    )

    print(
        "360 available:",
        event["freeze_frame"] is not None
    )

    break

consumer.close()

C:\Users\Kenda\AppData\Local\Temp\ipykernel_39068\2152779491.py:4: DeprecationWarning: value_deserializer does not implement kafka.serializer.Deserializer
  consumer = KafkaConsumer(


Event received successfully
Topic: football-events
Partition: 0
Offset: 0
Event: Pass | Team: Elche | Player: Pere Milla Peña
360 available: True


In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("FootballRealTimeAnalytics")
    .master("local[*]")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print("Spark version:", spark.version)
print("Spark session started successfully.")

c:\Users\Kenda\AppData\Local\Programs\Python\Python314\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Spark version: 4.2.0
Spark session started successfully.


In [2]:
spark.stop()

In [3]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("FootballRealTimeAnalytics")
    .master("local[*]")
    .config(
        "spark.jars.packages",
        "org.apache.spark:spark-sql-kafka-0-10_2.13:4.2.0"
    )
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print("Spark version:", spark.version)
print("Spark with Kafka connector started.")

Spark version: 4.2.0
Spark with Kafka connector started.


In [5]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("FootballRealTimeAnalytics")
    .master("local[*]")
    .config(
        "spark.jars.packages",
        "org.apache.spark:spark-sql-kafka-0-10_2.13:4.2.0"
    )
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print("Spark version:", spark.version)
print(
    "Kafka package:",
    spark.sparkContext.getConf().get("spark.jars.packages")
)

Spark version: 4.2.0
Kafka package: org.apache.spark:spark-sql-kafka-0-10_2.13:4.2.0


In [7]:
import os

os.environ["PYSPARK_SUBMIT_ARGS"] = (
    "--packages "
    "org.apache.spark:spark-sql-kafka-0-10_2.13:4.2.0 "
    "pyspark-shell"
)

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("FootballRealTimeAnalytics")
    .master("local[*]")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print("Spark version:", spark.version)
print("Spark started with Kafka package request.")

Spark version: 4.2.0
Spark started with Kafka package request.


In [10]:
import pandas as pd

historical_master = pd.read_pickle(
    "historical_player_match_master_corrected.pkl"
)

In [11]:
match_id = 3764440

historical_master["match_id"] = pd.to_numeric(
    historical_master["match_id"],
    errors="coerce"
).astype("Int64")

match_history = historical_master[
    historical_master["match_id"] == match_id
].copy()

print("Historical rows:", len(match_history))
print("Players:", match_history["player_id"].nunique())

Historical rows: 32
Players: 32


In [12]:
history_lookup = (
    match_history
    .set_index("player_id")
    .to_dict("index")
)

print("Players in history lookup:", len(history_lookup))

Players in history lookup: 32


In [ ]:
import json

with open(
    "open-data-master/data/events/3764440.json",
    "r",
    encoding="utf-8"
) as file:
    events = json.load(file)

first_half_end_events = [
    event
    for event in events
    if (
        event["period"] == 1
        and event["type"]["name"] == "Half End"
    )
]

for event in first_half_end_events:
    print(
        "Index:", event["index"],
        "| Period:", event["period"],
        "| Time:", event["timestamp"],
        "| Minute:", event["minute"],
        "| Second:", event["second"]
    )

Index: 1994 | Period: 1 | Time: 00:45:57.650 | Minute: 45 | Second: 57
Index: 1995 | Period: 1 | Time: 00:45:57.650 | Minute: 45 | Second: 57


: 